# Can we predict late deliveries?
Protocol: `protocol.yaml` (temporal holdout from `holdout.csv`, 5-split walk-forward CV on training rows, selection by mean CV ROC AUC).

**Feature decision.** `claim_paid_eur` is compensation paid on a claim settled for the consignment, so it is only known after delivery (it is >0 almost only for late ones). It is excluded as target leakage. `consignment` (identifier) and `dispatched_on` (split only) are excluded per the protocol.

In [ ]:
import json, hashlib, numpy as np, pandas as pd
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

assert hashlib.sha256(open('data.csv','rb').read()).hexdigest()=='859a474044a292565d72b06a6180ade1c1adbc8c72a47c4a518186876f581f27'
assert hashlib.sha256(open('holdout.csv','rb').read()).hexdigest()=='fae5c02170970fe92fcadaee6d1109bb1d459c9dc033dfc487bb2c823d0f6736'
df=pd.read_csv('data.csv')
hold_idx=pd.read_csv('holdout.csv')['row'].to_numpy()
assert len(df)==8051 and len(hold_idx)==1610
train_idx=np.setdiff1d(np.arange(len(df)),hold_idx)
df['t']=pd.to_datetime(df['dispatched_on'])
train=df.iloc[train_idx].sort_values('t',kind='stable')   # time-ordered for walk-forward CV
hold=df.iloc[hold_idx]
print(len(train),len(hold),train.t.max(),hold.t.min())

In [ ]:
cat=['carrier']; num=['distance_km','weight_kg','parcels']
features=cat+num
X_tr,y_tr=train[features],train['late_delivery']
X_ho,y_ho=hold[features],hold['late_delivery']
def pre(scale):
    return ColumnTransformer([('c',OneHotEncoder(handle_unknown='ignore'),cat),
                              ('n',StandardScaler() if scale else 'passthrough',num)])
# all candidates named before any is scored
candidates={
 'baseline_prior':Pipeline([('p',pre(False)),('m',DummyClassifier(strategy='prior'))]),
 'logreg_C0.1':Pipeline([('p',pre(True)),('m',LogisticRegression(C=0.1,max_iter=1000))]),
 'logreg_C1':Pipeline([('p',pre(True)),('m',LogisticRegression(C=1,max_iter=1000))]),
 'random_forest':Pipeline([('p',pre(False)),('m',RandomForestClassifier(300,min_samples_leaf=20,random_state=0,n_jobs=-1))]),
 'hist_gb':Pipeline([('p',pre(False)),('m',HistGradientBoostingClassifier(max_depth=3,learning_rate=0.05,max_iter=200,random_state=0))]),
}
print(list(candidates))

In [ ]:
folds=list(TimeSeriesSplit(n_splits=5).split(X_tr))
cv_scores={}
for name,m in candidates.items():
    s=[]
    for a,b in folds:   # preprocessing lives in the pipeline, so it is refit per fold
        mm=clone(m).fit(X_tr.iloc[a],y_tr.iloc[a])
        s.append(roc_auc_score(y_tr.iloc[b],mm.predict_proba(X_tr.iloc[b])[:,1]))
    cv_scores[name]=s
print(pd.DataFrame({k:{'mean':np.mean(v),'std':np.std(v)} for k,v in cv_scores.items()}).T)
chosen=max(cv_scores,key=lambda k:np.mean(cv_scores[k]))
print('chosen:',chosen)

In [ ]:
final=clone(candidates[chosen]).fit(X_tr,y_tr)
holdout_auc=roc_auc_score(y_ho,final.predict_proba(X_ho)[:,1])   # scored once
print('holdout roc_auc',holdout_auc)
pos=lambda idx:[int(i) for i in idx]
evidence={
 'features':features,
 'candidates':list(candidates),
 'cv_folds':[[pos(train.index[a]),pos(train.index[b])] for a,b in folds],
 'cv_scores':cv_scores,
 'chosen':chosen,
 'holdout_rows':pos(hold_idx),
 'holdout_scores':{chosen:holdout_auc},
}
json.dump(evidence,open('evidence.json','w'))

## Conclusion
The headline is the holdout ROC AUC of the CV-chosen candidate (printed above). `claim_paid_eur` was deliberately excluded: it would make the task look trivially predictable but is not available at prediction time.